# 00 — Prepare Labels & Enhanced Train Audio

Run this notebook **once** on Kaggle. Outputs go to `/kaggle/working/` and should be saved as Kaggle Dataset `vbd-labels-enhanced-train`.

**Two label sources (switchable):**
- `LABEL_SOURCE = 'official'` (default) — download the original VoiceBank prompts from Edinburgh DataShare (`testset_txt.zip` + `trainset_28spk_txt.zip`). Fast (~30 s), authoritative.
- `LABEL_SOURCE = 'whisper'` — transcribe the **clean** audio with `whisper-large-v3` to produce pseudo-labels. Use this if the download fails or you suspect prompt/audio mismatch.
- `LABEL_SOURCE = 'auto'` — try official first; on any failure fall back to whisper automatically.

**Produces** in `/kaggle/working/`:
- `transcripts.csv` — columns `id, split, text, source`.
- `enhanced_train/<id>.wav` — MetricGAN+ enhancement of the **noisy train** split.
- `enhanced_train_manifest.csv`.

The noisy test set is **not** enhanced — both flows in notebooks 01/02 are evaluated on raw noisy test audio.

In [1]:
LABEL_SOURCE = 'auto'   # 'official' | 'whisper' | 'auto'
RUN_ENHANCEMENT = True  # set False if you only want transcripts (e.g. for Flow A only)
# CHANGE THIS LINE to match the path you saw in /kaggle/input/.
# Kaggle uses the dataset's URL slug, which may differ from the display name.
# Examples of what it might actually be:
#   /kaggle/input/vbd-official-transcripts
#   /kaggle/input/voicebank-transcripts
#   /kaggle/input/voicebank-official-txt
KAGGLE_OFFICIAL_DIR = '/kaggle/input/datasets/tejaswagg/vbd-official-transcripts'

In [2]:
from pathlib import Path
root = Path(KAGGLE_OFFICIAL_DIR)
print('exists:', root.exists())
print('contents:', list(root.iterdir()) if root.exists() else 'NOT FOUND')

exists: True
contents: [PosixPath('/kaggle/input/datasets/tejaswagg/vbd-official-transcripts/trainset_28spk_txt'), PosixPath('/kaggle/input/datasets/tejaswagg/vbd-official-transcripts/testset_txt')]


In [3]:
!pip install -q transformers==4.46.* accelerate datasets speechbrain soundfile librosa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 48.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is i

In [4]:
import os, gc, io, json, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import soundfile as sf

assert torch.cuda.is_available(), 'Enable GPU T4 x1 in Kaggle settings.'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True, parents=True)
ENH_DIR = WORK / 'enhanced_train'; ENH_DIR.mkdir(exist_ok=True)
MODEL_DIR = WORK / 'models'; MODEL_DIR.mkdir(exist_ok=True)
TXT_DIR = WORK / 'official_txt'; TXT_DIR.mkdir(exist_ok=True)
SR = 16000
print('device:', torch.cuda.get_device_name(0))

device: Tesla T4


In [5]:
from datasets import load_dataset
ds = load_dataset('JacobLinCool/VoiceBank-DEMAND-16k')
train_ds, test_ds = ds['train'], ds['test']
print('train:', len(train_ds), '| test:', len(test_ds))
train_ids = {s['id'] for s in train_ds}
test_ids  = {s['id'] for s in test_ds}

README.md:   0%|          | 0.00/652 [00:00<?, ?B/s]

data/train-00000-of-00005.parquet:   0%|          | 0.00/444M [00:00<?, ?B/s]

data/train-00001-of-00005.parquet:   0%|          | 0.00/416M [00:00<?, ?B/s]

data/train-00002-of-00005.parquet:   0%|          | 0.00/431M [00:00<?, ?B/s]

data/train-00003-of-00005.parquet:   0%|          | 0.00/444M [00:00<?, ?B/s]

data/train-00004-of-00005.parquet:   0%|          | 0.00/416M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/132M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11572 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/824 [00:00<?, ? examples/s]

train: 11572 | test: 824


## 1. Try the official Edinburgh DataShare transcripts

URLs verified from https://datashare.ed.ac.uk/items/6ed35425-bf14-4d2b-93a1-0a4984952757

Each zip contains `*.txt` files like `p226_001.txt` holding the prompt text. The HF dataset IDs match these stems directly.

In [6]:
OFFICIAL_URLS = {
    'test':  'https://datashare.ed.ac.uk/bitstreams/09e12592-879e-4749-bf6d-e34cf0d1ddf6/download',
    'train': 'https://datashare.ed.ac.uk/bitstreams/fb55b5de-c2db-4451-ac0a-134e98011ffa/download',
}


def _classify_split(stem: str) -> str | None:
    if stem in train_ids:
        return 'train'
    if stem in test_ids:
        return 'test'
    return None


def _read_text(raw: bytes) -> str:
    return raw.decode('utf-8', errors='ignore').strip()


def fetch_kaggle_official(root: str) -> pd.DataFrame | None:
    root_path = Path(root)
    if not root_path.exists():
        print(f'  KAGGLE_OFFICIAL_DIR not found: {root_path}')
        return None

    print(f'  scanning {root_path} ...')
    all_files = list(root_path.rglob('*'))
    txts = [p for p in all_files if p.suffix == '.txt']
    zips = [p for p in all_files if p.suffix == '.zip']
    print(f'  entries: {len(all_files)} | .txt files: {len(txts)} | .zip files: {len(zips)}')
    for z in zips[:5]:
        print(f'    zip: {z}')
    for t in txts[:3]:
        print(f'    txt: {t}')

    rows, seen, unmatched = [], set(), []

    for txt in txts:
        sid = txt.stem
        split = _classify_split(sid)
        if split is None:
            if len(unmatched) < 20:
                unmatched.append(sid)
            continue
        if sid in seen:
            continue
        rows.append({'id': sid, 'split': split,
                     'text': _read_text(txt.read_bytes()),
                     'source': 'official-kaggle'})
        seen.add(sid)

    for zp in zips:
        try:
            with zipfile.ZipFile(zp) as zf:
                members = [n for n in zf.namelist() if n.endswith('.txt')]
                print(f'  {zp.name}: {len(members)} .txt members')
                for m in members:
                    sid = Path(m).stem
                    split = _classify_split(sid)
                    if split is None:
                        if len(unmatched) < 20:
                            unmatched.append(sid)
                        continue
                    if sid in seen:
                        continue
                    with zf.open(m) as fh:
                        rows.append({'id': sid, 'split': split,
                                     'text': _read_text(fh.read()),
                                     'source': 'official-kaggle'})
                        seen.add(sid)
        except Exception as e:
            print(f'  zip read failed for {zp}: {e}')

    if unmatched:
        print(f'  sample unmatched stems: {unmatched[:5]}')
        print(f'  sample HF train ids:   {list(train_ids)[:3]}')
        print(f'  sample HF test ids:    {list(test_ids)[:3]}')

    if not rows:
        print('  No matching .txt files found in KAGGLE_OFFICIAL_DIR.')
        return None

    df = pd.DataFrame(rows)
    n_tr = (df['split'] == 'train').sum()
    n_te = (df['split'] == 'test').sum()
    print(f'  Kaggle official: matched train {n_tr}/{len(train_ids)}, test {n_te}/{len(test_ids)}')
    if n_tr < 0.5 * len(train_ids) or n_te < 0.5 * len(test_ids):
        print('  coverage <50% — returning None so we fall through to next source.')
        return None
    if n_tr < 0.95 * len(train_ids) or n_te < 0.95 * len(test_ids):
        print('  coverage <95% — proceeding anyway (training will drop ids without transcripts).')
    return df


def fetch_official() -> pd.DataFrame | None:
    rows = []
    for split, url in OFFICIAL_URLS.items():
        target_ids = train_ids if split == 'train' else test_ids
        zip_path = TXT_DIR / f'{split}.zip'
        print(f'  downloading {split} -> {zip_path}')
        try:
            urllib.request.urlretrieve(url, zip_path)
            with zipfile.ZipFile(zip_path) as zf:
                txt_members = [n for n in zf.namelist() if n.endswith('.txt')]
                print(f'    {len(txt_members)} .txt files in zip')
                found = 0
                for m in txt_members:
                    sid = Path(m).stem
                    if sid not in target_ids:
                        continue
                    with zf.open(m) as fh:
                        text = _read_text(fh.read())
                    rows.append({'id': sid, 'split': split, 'text': text, 'source': 'official'})
                    found += 1
                print(f'    matched {found}/{len(target_ids)} ids for {split}')
                if found < 0.95 * len(target_ids):
                    raise RuntimeError(f'Only matched {found}/{len(target_ids)} ids in {split}.')
        except Exception as e:
            print(f'    FAILED for {split}: {e}')
            return None
    return pd.DataFrame(rows)


official_df = None

if LABEL_SOURCE in ('kaggle', 'auto'):
    print(f'Trying Kaggle-uploaded transcripts at {KAGGLE_OFFICIAL_DIR} ...')
    official_df = fetch_kaggle_official(KAGGLE_OFFICIAL_DIR)
    if official_df is None and LABEL_SOURCE == 'kaggle':
        raise RuntimeError(f'LABEL_SOURCE=kaggle but no usable transcripts in {KAGGLE_OFFICIAL_DIR}.')

if official_df is None and LABEL_SOURCE in ('official', 'auto'):
    print('Attempting official transcript download from Edinburgh DataShare...')
    official_df = fetch_official()
    if official_df is None and LABEL_SOURCE == 'official':
        raise RuntimeError('Official download failed and LABEL_SOURCE=official.')

if official_df is not None:
    print('Official transcripts OK:', len(official_df), 'rows | source:', official_df['source'].unique())
    print(official_df.head())
else:
    print('No official transcripts available — will fall back to whisper path.')

Trying Kaggle-uploaded transcripts at /kaggle/input/datasets/tejaswagg/vbd-official-transcripts ...
  scanning /kaggle/input/datasets/tejaswagg/vbd-official-transcripts ...
  entries: 12400 | .txt files: 12396 | .zip files: 0
    txt: /kaggle/input/datasets/tejaswagg/vbd-official-transcripts/trainset_28spk_txt/trainset_28spk_txt/p276_353.txt
    txt: /kaggle/input/datasets/tejaswagg/vbd-official-transcripts/trainset_28spk_txt/trainset_28spk_txt/p287_239.txt
    txt: /kaggle/input/datasets/tejaswagg/vbd-official-transcripts/trainset_28spk_txt/trainset_28spk_txt/p228_362.txt
  Kaggle official: matched train 11572/11572, test 824/824
Official transcripts OK: 12396 rows | source: ['official-kaggle']
         id  split                                               text  \
0  p276_353  train         The move follows a review of the Singapore   
1  p287_239  train                       The effects are not all bad.   
2  p228_362  train        It took about an hour for the gas to clear.   
3  

## 2. Fallback path — whisper-large-v3 pseudo-labels from clean audio

Runs only if `LABEL_SOURCE == 'whisper'` OR the official download failed under `'auto'`.

In [7]:
use_whisper = (LABEL_SOURCE == 'whisper') or (LABEL_SOURCE == 'auto' and official_df is None)

if use_whisper:
    from transformers import pipeline
    # whisper-medium.en (English-only, ~3x smaller than large-v3).
    # NOTE: do NOT pass `no_speech_threshold` without `logprob_threshold` —
    # transformers 4.46 has a bug where the fallback path UnboundLocalErrors on `logprobs`.
    WHISPER_MODEL = 'openai/whisper-medium.en'
    BATCH = 8

    asr = pipeline(
        'automatic-speech-recognition',
        model=WHISPER_MODEL,
        torch_dtype=torch.float16,
        device='cuda',
        chunk_length_s=30,
        return_timestamps=False,
        batch_size=BATCH,
    )

    def _audio_stream(hf_split):
        for s in hf_split:
            yield {
                'raw': np.asarray(s['clean']['array'], dtype=np.float32),
                'sampling_rate': SR,
            }

    def transcribe_split(hf_split, split_name):
        ids = [s['id'] for s in hf_split]
        rows = []
        for idx, out in enumerate(asr(_audio_stream(hf_split), batch_size=BATCH)):
            rows.append({
                'id': ids[idx],
                'split': split_name,
                'text': (out['text'] or '').strip(),
                'source': WHISPER_MODEL.split('/')[-1],
            })
            if (idx + 1) % 200 == 0:
                print(f'  {split_name} {idx+1}/{len(ids)}')
                torch.cuda.empty_cache()
        return rows

    whisper_rows = transcribe_split(test_ds, 'test') + transcribe_split(train_ds, 'train')
    whisper_df = pd.DataFrame(whisper_rows)
    print('whisper rows:', len(whisper_df))
    del asr; gc.collect(); torch.cuda.empty_cache()
else:
    whisper_df = None
    print('Whisper path skipped — using official transcripts.')

Whisper path skipped — using official transcripts.


In [8]:
tr_df = official_df if official_df is not None else whisper_df
tr_df.to_csv(WORK/'transcripts.csv', index=False)
print('saved transcripts.csv:', len(tr_df), 'rows | source(s):', tr_df['source'].unique())
tr_df.head()

saved transcripts.csv: 12396 rows | source(s): ['official-kaggle']


,id,split,text,source
0,p276_353,train,The move follows a review of the Singapore,official-kaggle
1,p287_239,train,The effects are not all bad.,official-kaggle
2,p228_362,train,It took about an hour for the gas to clear.,official-kaggle
3,p282_095,train,"Yet, it worked.",official-kaggle
4,p287_285,train,Both companies have had a rough ride in recent...,official-kaggle


In [9]:
if RUN_ENHANCEMENT:
    from speechbrain.inference.enhancement import SpectralMaskEnhancement
    enhancer = SpectralMaskEnhancement.from_hparams(
        source='speechbrain/metricgan-plus-voicebank',
        savedir=str(MODEL_DIR/'metricgan-plus'),
        run_opts={'device': 'cuda'},
    )
    manifest = []
    for i, s in enumerate(train_ds):
        sid = s['id']
        out_path = ENH_DIR / f'{sid}.wav'
        if out_path.exists():
            manifest.append({'id': sid, 'path': str(out_path)})
            continue
        noisy = torch.tensor(s['noisy']['array'], dtype=torch.float32).unsqueeze(0)
        lengths = torch.tensor([1.0])
        with torch.no_grad():
            enh = enhancer.enhance_batch(noisy.cuda(), lengths=lengths.cuda())
        enh_np = enh.squeeze().cpu().numpy().astype(np.float32)
        sf.write(out_path, enh_np, SR, subtype='PCM_16')
        manifest.append({'id': sid, 'path': str(out_path)})
        if (i+1) % 500 == 0:
            print(f'  enhanced {i+1}/{len(train_ds)}')
            torch.cuda.empty_cache()
    mf = pd.DataFrame(manifest)
    mf.to_csv(WORK/'enhanced_train_manifest.csv', index=False)
    print('manifest rows:', len(mf))
else:
    print('RUN_ENHANCEMENT=False — skipping. Flow B notebook will fail without these files.')

hyperparams.yaml: 0.00B [00:00, ?B/s]

enhance_model.ckpt:   0%|          | 0.00/7.59M [00:00<?, ?B/s]

Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


  enhanced 500/11572
  enhanced 1000/11572
  enhanced 1500/11572
  enhanced 2000/11572
  enhanced 2500/11572
  enhanced 3000/11572
  enhanced 3500/11572
  enhanced 4000/11572
  enhanced 4500/11572
  enhanced 5000/11572
  enhanced 5500/11572
  enhanced 6000/11572
  enhanced 6500/11572
  enhanced 7000/11572
  enhanced 7500/11572
  enhanced 8000/11572
  enhanced 8500/11572
  enhanced 9000/11572
  enhanced 9500/11572
  enhanced 10000/11572
  enhanced 10500/11572
  enhanced 11000/11572
  enhanced 11500/11572
manifest rows: 11572


## 3. MetricGAN+ enhancement of the TRAIN noisy split

Skip by setting `RUN_ENHANCEMENT = False` at the top.

In [10]:
# Sanity: spot-check a couple of enhanced WAVs and their (now-known) transcripts
from IPython.display import Audio, display
lookup = dict(zip(tr_df['id'], tr_df['text']))
for sid in [train_ds[0]['id'], train_ds[100]['id']]:
    print(sid, '|', lookup.get(sid, '<no transcript>')[:80])
    wav = ENH_DIR/f'{sid}.wav'
    if wav.exists():
        display(Audio(str(wav)))

p226_001 | Please call Stella.


p226_105 | May not have been at fault?


## 4. Done — turn /kaggle/working into a dataset

1. `Save Version → Save & Run All`.
2. Output tab → **Create new dataset** named `vbd-labels-enhanced-train`.
3. Add that dataset as an input to notebooks 01, 02, 03.

In [11]:
for p in sorted(WORK.glob('*')):
    if p.is_dir():
        n = sum(1 for _ in p.iterdir())
        print(f'{p.name}/  ({n} entries)')
    else:
        print(f'{p.name}  ({p.stat().st_size//1024} KB)')

__notebook__.ipynb  (238 KB)
enhanced_train/  (11572 entries)
enhanced_train_manifest.csv  (598 KB)
models/  (1 entries)
official_txt/  (0 entries)
transcripts.csv  (873 KB)
